In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error
import optuna


In [3]:
# Liste des colonnes d'intérêt
colonnes_utiles = [
    'release_date',       # Date
    'duration',           # Float
    "director",           # Categorical
    'distributor',        # Categorical
    'movie_type',         # Categorical
    'languages',          # Categorical
    'audience',           # Categorical
    'Pays',               # Categorical
    'max_moyenne',        # Float
    'cluster_acteur_principal',  # Categorical
    'budget',             # Float
    'box_office_fr'       # cible (target)
]

# Chargement des données et sélection des colonnes utiles
data = pd.read_csv("xgboost80000.csv")
data = data[colonnes_utiles].copy()


In [4]:
# Convertir 'release_date' en datetime
data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')

# Extraire des informations temporelles
data['release_year'] = data['release_date'].dt.year
data['release_month'] = data['release_date'].dt.month

# Vous pouvez ensuite décider de conserver ou de supprimer la colonne d'origine
data.drop(columns=['release_date'], inplace=True)


In [5]:
colonnes_categorielles = [
    "director", "distributor", "movie_type", "languages",
    "audience", "Pays", "cluster_acteur_principal"
]

for col in colonnes_categorielles:
    if col in data.columns:
        data[col] = data[col].astype('category')


In [6]:
print(data.isnull().sum())

# Par exemple, pour remplir les valeurs manquantes numériques avec la médiane :
for col in ['duration', 'max_moyenne', 'budget']:
    data[col].fillna(data[col].median(), inplace=True)

# Pour les colonnes catégorielles, vous pouvez remplir avec une chaîne "Inconnu" ou conserver la catégorie manquante
for col in colonnes_categorielles:
    data[col].fillna("Inconnu", inplace=True)
    data[col] = data[col].astype('category')


duration                       0
director                      15
distributor                    0
movie_type                     0
languages                      0
audience                    1920
Pays                         374
max_moyenne                    0
cluster_acteur_principal     128
budget                       928
box_office_fr                  0
release_year                   0
release_month                  0
dtype: int64


/tmp/ipykernel_48223/4191548258.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].median(), inplace=True)
/tmp/ipykernel_48223/4191548258.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

TypeError: Cannot setitem on a Categorical with a new category (Inconnu), set the categories first